# A reproducible model-evaluation record

Evaluate a deliberately simple deterministic classifier, keep the examples visible, and produce a provenance record. The goal is measurement discipline—not a performance claim.

In [ ]:
import hashlib
import json
import math
import platform
from datetime import datetime, timezone

## Freeze the examples and prediction rule

All labels and inputs are synthetic. The keyword rule is intentionally inspectable and deterministic, so the same notebook produces the same result without model downloads or hidden services.

In [ ]:
examples = [
    {"text": "the release checks passed", "label": 1},
    {"text": "verification failed safely", "label": 0},
    {"text": "tests passed after review", "label": 1},
    {"text": "the scanner failed closed", "label": 0},
    {"text": "review remains pending", "label": 0},
    {"text": "all required checks passed", "label": 1},
]

def predict(text):
    return int("passed" in text.lower())

predictions = [predict(row["text"]) for row in examples]
assert predictions == [1, 0, 1, 0, 0, 1]

In [ ]:
tp = sum(pred == 1 and row["label"] == 1 for pred, row in zip(predictions, examples))
tn = sum(pred == 0 and row["label"] == 0 for pred, row in zip(predictions, examples))
fp = sum(pred == 1 and row["label"] == 0 for pred, row in zip(predictions, examples))
fn = sum(pred == 0 and row["label"] == 1 for pred, row in zip(predictions, examples))
accuracy = (tp + tn) / len(examples)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
metrics = {
    "sample_count": len(examples),
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "confusion_matrix": {"tp": tp, "tn": tn, "fp": fp, "fn": fn},
}
print(json.dumps(metrics, indent=2))

## Report uncertainty and boundaries

Six synthetic examples cannot support a general model-quality claim. A Wilson interval makes the sampling uncertainty visible, but it does not correct dataset bias, leakage, label errors, or distribution shift.

In [ ]:
z = 1.96
n = len(examples)
center = (accuracy + z*z/(2*n)) / (1 + z*z/n)
margin = z * math.sqrt(accuracy*(1-accuracy)/n + z*z/(4*n*n)) / (1 + z*z/n)
metrics["accuracy_wilson_95"] = [max(0.0, center - margin), min(1.0, center + margin)]
print(json.dumps(metrics, indent=2))

In [ ]:
canonical_examples = json.dumps(examples, sort_keys=True, separators=(",", ":"))
record = {
    "schema_version": 1,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "dataset_sha256": hashlib.sha256(canonical_examples.encode()).hexdigest(),
    "prediction_rule": "case-insensitive substring: passed",
    "python": platform.python_version(),
    "metrics": metrics,
    "claim_boundary": "Synthetic tutorial smoke test; not evidence of general model quality.",
}
print(json.dumps(record, indent=2))